# BipedalWalker — Backflip Curriculum (v2)

Trains a BipedalWalker agent to perform clean backflips and land on its feet.

## Three-Stage Curriculum

| Stage | Goal | Gravity | Fall penalty | Key rewards |
|-------|------|---------|--------------|-------------|
| 1 | Learn to flip | −5.0 (easy) | Cancelled | Angular speed, rotation progress, milestones |
| 2 | Learn to land upright | −7.5 | Cancelled pre-flip only | Uprightness gradient, knee-crash penalty, clean-landing bonus |
| 3 | Consolidate under real gravity | −10.0 (real) | Not cancelled | Same as Stage 2, higher stakes |

## Key Fixes vs the Original Wrapper
- **Gravity curriculum** — gravity increases each stage toward real physics
- **Fixed landing check** — landing bonus only fires when hull angle < 0.4 rad (upright)
- **Knee-crash penalty** — landing with hull > 0.8 rad gives −100 and terminates the episode
- **Uprightness gradient** — `cos(hull_angle) × 25` guides agent back to vertical post-flip
- **Spin dampening** — angular velocity is penalised after flip completes (stop spinning, land!)
- **In-air tuck reward** — bent knees while descending help prepare for feet-first landing
- **Rotation reward stops after flip** — no incentive to keep spinning after 360°

## How to Run
Run cells top to bottom. Each stage saves a `.zip` model and the next stage loads it.
You can restart from any stage if a save already exists — the training cell will skip.

To monitor training live:
```
tensorboard --logdir ./tb_logs_flipper
```

## 1. Imports & Setup

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import gymnasium as gym
import numpy as np
import torch

import custom_bipedal  # your local copy of the BipedalWalker environment

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback, CallbackList

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
NUM_ENVS = 32
print(f"Device: {DEVICE}  |  Parallel envs: {NUM_ENVS}")

Device: cuda  |  Parallel envs: 32


## 2. Improved `CurriculumFlipperWrapper`

Run this cell once — every training/test cell below depends on it.

In [2]:
class CurriculumFlipperWrapper(gym.Wrapper):
    """
    3-stage curriculum wrapper for BipedalWalker backflip training.

    Stage 1 — Rotation Mastery  (gravity = -5.0)
        Learn to discover and reliably complete a full backflip.
        Fall penalty is cancelled so the agent takes risks.

    Stage 2 — Landing  (gravity = -7.5)
        Learn to land upright on feet after the flip.
        Knee-crash landings are penalised and terminate the episode.
        Clean landings give a big bonus.

    Stage 3 — Consolidation  (gravity = -10.0, real physics)
        Same as Stage 2 under full gravity with no hand-holding.
    """

    # Gravity per stage — increases toward real gravity each stage
    GRAVITY = {1: -5.0, 2: -7.5, 3: -10.0}

    # Landing quality thresholds (radians, 0 = perfectly upright)
    CLEAN_LANDING_ANGLE = 0.4   # ~23° off vertical → success
    CRASH_LANDING_ANGLE = 0.8   # ~46° off vertical → knee crash → failure

    def __init__(self, env, stage: int = 1, max_steps: int = 1500):
        super().__init__(env)
        self.stage     = stage
        self.max_steps = max_steps

        # Per-episode state — reset in reset()
        self.cumulative_angle = 0.0
        self.prev_angle       = 0.0
        self.flip_completed   = False
        self.landed           = False
        self.step_counter     = 0
        self._milestone_flags: dict = {}

        # Append normalised flip progress to observation
        low  = np.append(self.env.observation_space.low,  -np.inf)
        high = np.append(self.env.observation_space.high,  np.inf)
        self.observation_space = gym.spaces.Box(low, high, dtype=np.float32)

    # ------------------------------------------------------------------
    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        # Stage-dependent gravity — closer to real each stage
        gravity = self.GRAVITY.get(self.stage, -5.0)
        try:
            self.env.unwrapped.world.gravity = (0.0, float(gravity))
        except Exception:
            pass

        self.cumulative_angle = 0.0
        self.prev_angle       = obs[0]
        self.flip_completed   = False
        self.landed           = False
        self.step_counter     = 0
        self._milestone_flags = {}

        return np.append(obs, 0.0).astype(np.float32), info

    # ------------------------------------------------------------------
    def step(self, action):
        obs, base_reward, terminated, truncated, info = self.env.step(action)
        self.step_counter += 1

        # ── Angle tracking ──────────────────────────────────────────────
        current_angle = obs[0]
        delta_angle   = current_angle - self.prev_angle
        # Wrap to [-π, π] to handle the discontinuity at ±π
        if delta_angle >  np.pi: delta_angle -= 2 * np.pi
        if delta_angle < -np.pi: delta_angle += 2 * np.pi

        prev_cumulative        = self.cumulative_angle
        self.cumulative_angle += delta_angle
        self.prev_angle        = current_angle

        abs_angle = abs(self.cumulative_angle)
        abs_prev  = abs(prev_cumulative)

        # ── Convenient shorthands ────────────────────────────────────────
        hull_angle   = obs[0]           # ≈ 0 = upright, ±π = upside-down
        ang_vel      = obs[1]           # hull angular velocity
        foot1_down   = obs[8]  == 1.0
        foot2_down   = obs[13] == 1.0
        feet_contact = foot1_down or foot2_down
        in_air       = not foot1_down and not foot2_down
        is_falling   = (base_reward == -100)  # hull touched ground
        knee1_angle  = obs[6]           # 0 = extended, ~2 = tucked
        knee2_angle  = obs[11]

        custom_reward = 0.0

        # ══════════════════════════════════════════════════════════════
        # PRE-FLIP rewards (only until the flip is completed)
        # ══════════════════════════════════════════════════════════════
        if not self.flip_completed:
            # 1. Raw angular speed reward — spin faster!
            custom_reward += abs(ang_vel) * 5.0

            # 2. Monotone rotation progress
            #    Using abs() means wiggling back and forth yields NO progress
            custom_reward += (abs_angle - abs_prev) * 15.0

            # 3. Small airtime bonus during flip
            if in_air:
                custom_reward += 1.0

            # 4. Milestone bonuses: 25% / 50% / 75% of a full rotation
            for ms in [np.pi / 2, np.pi, 3 * np.pi / 2]:
                key = f"ms_{ms:.4f}"
                if not self._milestone_flags.get(key, False) and abs_angle >= ms:
                    self._milestone_flags[key] = True
                    custom_reward += 50.0

        # ══════════════════════════════════════════════════════════════
        # FLIP COMPLETION (full 2π rotation)
        # ══════════════════════════════════════════════════════════════
        if abs_angle >= 2 * np.pi and not self.flip_completed:
            self.flip_completed = True
            custom_reward += 300.0

        # ══════════════════════════════════════════════════════════════
        # POST-FLIP rewards (once flip is done, before landing)
        # ══════════════════════════════════════════════════════════════
        if self.flip_completed and not self.landed:

            # 1. Strong continuous uprightness reward
            #    cos(0)  = +1.0 → fully upright   (max reward)
            #    cos(π)  = -1.0 → fully inverted   (max penalty)
            #    This creates a smooth gradient that pulls the agent upright
            custom_reward += np.cos(hull_angle) * 25.0

            # 2. Dampen residual spin — stop rotating, prepare to land
            custom_reward -= abs(ang_vel) * 5.0

            # 3. In-air tuck reward: bent knees help control the descent
            if in_air:
                custom_reward += (knee1_angle + knee2_angle) * 1.5

            # 4. KNEE-CRASH PENALTY
            #    Feet touch ground BUT hull is still tilted / inverted
            #    → This is the main fix: previously ANY foot contact was rewarded,
            #      so the agent learned to flop onto its knees and collect the bonus
            if feet_contact and abs(hull_angle) > self.CRASH_LANDING_ANGLE:
                penalty = -100.0 if self.stage < 3 else -150.0
                custom_reward += penalty
                terminated = True   # failure — reset and try again

            # 5. CLEAN LANDING BONUS
            #    Feet touch ground AND hull is upright — success!
            elif feet_contact and abs(hull_angle) < self.CLEAN_LANDING_ANGLE:
                uprightness   = 1.0 - (abs(hull_angle) / self.CLEAN_LANDING_ANGLE)
                bonus         = 1000.0 if self.stage < 3 else 2000.0
                custom_reward += bonus * uprightness
                self.landed    = True
                terminated     = True   # success!

        # ══════════════════════════════════════════════════════════════
        # STAGE-SPECIFIC EXTRAS
        # ══════════════════════════════════════════════════════════════
        if self.stage == 1:
            # Cancel the hard -100 fall penalty entirely.
            # The agent must take risks to discover flipping at all.
            if is_falling:
                custom_reward += 100.0

        elif self.stage == 2:
            # Soften fall penalty only BEFORE the flip.
            # After the flip the agent should worry about landing, not falling.
            if is_falling and not self.flip_completed:
                custom_reward += 50.0

        elif self.stage == 3:
            pass  # Real gravity, no hand-holding

        # ── Step limit ──────────────────────────────────────────────────
        if self.step_counter >= self.max_steps:
            truncated = True

        # ── Info ────────────────────────────────────────────────────────
        info["flip_completed"]   = self.flip_completed
        info["landed"]           = self.landed
        info["cumulative_angle"] = self.cumulative_angle
        info["abs_angle_deg"]    = np.degrees(abs_angle)

        obs_out = np.append(obs, abs_angle / (2 * np.pi)).astype(np.float32)
        return obs_out, base_reward + custom_reward, terminated, truncated, info


print("CurriculumFlipperWrapper defined ✓")

CurriculumFlipperWrapper defined ✓


## 3. Callbacks

In [3]:
class FlipMetricsCallback(BaseCallback):
    """Logs flip/landing success rates to TensorBoard."""
    def __init__(self, window=200, verbose=0):
        super().__init__(verbose)
        self.window = window
        self._flip:    list = []
        self._landed:  list = []
        self._rot_deg: list = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "flip_completed" not in info:
                continue
            self._flip.append(float(info["flip_completed"]))
            self._landed.append(float(info.get("landed", False)))
            self._rot_deg.append(info.get("abs_angle_deg", 0.0))
            for buf in (self._flip, self._landed, self._rot_deg):
                if len(buf) > self.window:
                    buf.pop(0)
        if self._flip:
            self.logger.record("flip/success_rate", np.mean(self._flip))
            self.logger.record("flip/landing_rate", np.mean(self._landed))
            self.logger.record("flip/avg_rotation_deg", np.mean(self._rot_deg))
        return True


class RenderCallback(BaseCallback):
    """Periodically renders the current policy for visual inspection."""
    def __init__(self, render_freq=100_000, stage=1, verbose=0):
        super().__init__(verbose)
        self.render_freq = render_freq
        self.stage = stage
        self._env = None

    def _on_step(self) -> bool:
        if self.num_timesteps > 0 and self.num_timesteps % self.render_freq == 0:
            print(f"\n[Step {self.num_timesteps:,}] Rendering policy...")
            if self._env is None:
                e = custom_bipedal.BipedalWalker(render_mode="human")
                self._env = CurriculumFlipperWrapper(e, stage=self.stage)
            obs, _ = self._env.reset()
            done = False
            while not done:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, _, terminated, truncated, _ = self._env.step(action)
                done = terminated or truncated
            print("Render done. Resuming training.\n")
        return True


print("Callbacks defined ✓")

Callbacks defined ✓


## 4. Stage 1 — Rotation Mastery

**Gravity:** −5.0 (easy to get airborne and spin)  
**Goal:** reliably complete a full 360° backflip  
**Duration:** ~5 million steps  

If `ppo_flipper_stage1.zip` already exists, this cell is skipped automatically.

In [4]:
STAGE1_SAVE  = "ppo_flipper_stage1"
STAGE1_STEPS = 5_000_000

if os.path.exists(f"{STAGE1_SAVE}.zip"):
    print(f"Stage 1 model already exists ({STAGE1_SAVE}.zip) — skipping training.")
    print("Delete the file and re-run this cell to retrain from scratch.")
else:
    print("Stage 1: Rotation Mastery  (gravity = -5.0)")
    print("To monitor: tensorboard --logdir ./tb_logs_flipper\n")

    def make_env_s1():
        def _init():
            e = custom_bipedal.BipedalWalker(hardcore=False)
            return CurriculumFlipperWrapper(e, stage=1)
        return _init

    vec_env_s1 = SubprocVecEnv([make_env_s1() for _ in range(NUM_ENVS)])

    model_s1 = PPO(
        "MlpPolicy", vec_env_s1,
        verbose=1,
        device=DEVICE,
        n_steps=2048,
        batch_size=8192,
        n_epochs=5,
        learning_rate=3e-4,
        gae_lambda=0.95,
        gamma=0.99,
        clip_range=0.2,
        ent_coef=0.05,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
        tensorboard_log="./tb_logs_flipper",
    )

    cbs_s1 = CallbackList([
        FlipMetricsCallback(window=200)
    ])

    model_s1.learn(total_timesteps=STAGE1_STEPS, callback=cbs_s1, progress_bar=True)
    model_s1.save(STAGE1_SAVE)
    vec_env_s1.close()
    print(f"\nStage 1 complete. Saved to {STAGE1_SAVE}.zip")

Stage 1 model already exists (ppo_flipper_stage1.zip) — skipping training.
Delete the file and re-run this cell to retrain from scratch.


## 5. Stage 2 — Landing

**Gravity:** −7.5 (harder — landing sloppily actually hurts)  
**Goal:** land upright on feet after the flip  
**Duration:** ~4 million steps  

Loads Stage 1 weights and continues training with a lower learning rate.  
**New in this stage:** knee-crash penalty, clean-landing bonus, uprightness gradient.

In [ ]:
STAGE2_SAVE  = "ppo_flipper_stage2"
STAGE2_STEPS = 5_000_000

if os.path.exists(f"{STAGE2_SAVE}.zip"):
    print(f"Stage 2 model already exists ({STAGE2_SAVE}.zip) — skipping training.")
else:
    if not os.path.exists(f"{STAGE1_SAVE}.zip"):
        raise FileNotFoundError(f"Stage 1 model not found: {STAGE1_SAVE}.zip\nRun Stage 1 first.")

    print("Stage 2: Landing  (gravity = -7.5)")
    print("To monitor: tensorboard --logdir ./tb_logs_flipper\n")

    def make_env_s2():
        def _init():
            e = custom_bipedal.BipedalWalker(hardcore=False)
            return CurriculumFlipperWrapper(e, stage=2)
        return _init

    vec_env_s2 = SubprocVecEnv([make_env_s2() for _ in range(NUM_ENVS)])

    # Load Stage 1 weights, plug into Stage 2 env
    model_s2 = PPO.load(
        STAGE1_SAVE, env=vec_env_s2, device=DEVICE,
        tensorboard_log="./tb_logs_flipper"
    )
    # Fine-tune with lower LR and less entropy
    model_s2.learning_rate = 1e-4
    model_s2.ent_coef      = 0.005

    cbs_s2 = CallbackList([
        FlipMetricsCallback(window=200)
    ])

    model_s2.learn(
        total_timesteps=STAGE2_STEPS, callback=cbs_s2,
        progress_bar=True, reset_num_timesteps=False
    )
    model_s2.save(STAGE2_SAVE)
    vec_env_s2.close()
    print(f"\nStage 2 complete. Saved to {STAGE2_SAVE}.zip")

## 6. Stage 3 — Consolidation (Real Gravity)

**Gravity:** −10.0 (real Earth gravity)  
**Goal:** perform and land clean backflips under realistic physics  
**Duration:** ~3 million steps  

No fall penalty assistance. Higher stakes for both success and failure.

In [ ]:
STAGE2_SAVE = "ppo_flipper_stage2"
STAGE3_SAVE  = "ppo_flipper_stage3"
STAGE3_STEPS = 5_000_000

if os.path.exists(f"{STAGE3_SAVE}.zip"):
    print(f"Stage 3 model already exists ({STAGE3_SAVE}.zip) — skipping training.")
else:
    if not os.path.exists(f"{STAGE2_SAVE}.zip"):
        raise FileNotFoundError(f"Stage 2 model not found: {STAGE2_SAVE}.zip\nRun Stage 2 first.")

    print("Stage 3: Real-gravity consolidation  (gravity = -10.0)")
    print("To monitor: tensorboard --logdir ./tb_logs_flipper\n")

    def make_env_s3():
        def _init():
            e = custom_bipedal.BipedalWalker(hardcore=False)
            return CurriculumFlipperWrapper(e, stage=3)
        return _init

    vec_env_s3 = SubprocVecEnv([make_env_s3() for _ in range(NUM_ENVS)])

    model_s3 = PPO.load(
        STAGE2_SAVE, env=vec_env_s3, device=DEVICE,
        tensorboard_log="./tb_logs_flipper"
    )
    model_s3.learning_rate = 5e-5
    model_s3.ent_coef      = 0.001

    cbs_s3 = CallbackList([
        FlipMetricsCallback(window=200)
    ])

    model_s3.learn(
        total_timesteps=STAGE3_STEPS, callback=cbs_s3,
        progress_bar=True, reset_num_timesteps=False
    )
    model_s3.save(STAGE3_SAVE)
    vec_env_s3.close()
    print(f"\nStage 3 complete. Saved to {STAGE3_SAVE}.zip")

## 7. Test the Final Model

Loads the best available saved model and runs 5 rendered test episodes.  
Priority: Stage 3 > Stage 2 > Stage 1

In [8]:
import time

# Pick the best available model
for path, stage in [("ppo_flipper_stage3", 3), ("ppo_flipper_stage2", 2), ("ppo_flipper_stage1", 1)]:
    if os.path.exists(f"{path}.zip"):
        model_path, model_stage = path, stage
        break
else:
    raise FileNotFoundError("No saved model found. Train at least Stage 1 first.")

print(f"Loading model: {model_path}.zip  (Stage {model_stage})")
model_test = PPO.load(model_path, device=DEVICE)

env_test = custom_bipedal.BipedalWalker(hardcore=False, render_mode="human")
env_test = CurriculumFlipperWrapper(env_test, stage=model_stage)

NUM_TEST_EPISODES = 5

for ep in range(1, NUM_TEST_EPISODES + 1):
    obs, _ = env_test.reset()
    done = False
    total_reward = 0.0
    steps = 0

    while not done:
        action, _ = model_test.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env_test.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1

    print(f"Episode {ep:2d}  |  Steps: {steps:4d}  |  Reward: {total_reward:8.1f}  |  "
          f"Flip: {'✓' if info['flip_completed'] else '✗'}  |  "
          f"Landed: {'✓' if info['landed'] else '✗'}  |  "
          f"Rotation: {info['abs_angle_deg']:.1f}°")
    time.sleep(0.5)

env_test.close()

Loading model: ppo_flipper_stage3.zip  (Stage 3)
Episode  1  |  Steps:   92  |  Reward:    251.8  |  Flip: ✗  |  Landed: ✗  |  Rotation: 338.5°
Episode  2  |  Steps:   92  |  Reward:    251.9  |  Flip: ✗  |  Landed: ✗  |  Rotation: 337.0°
Episode  3  |  Steps:   84  |  Reward:    167.5  |  Flip: ✗  |  Landed: ✗  |  Rotation: 232.3°
Episode  4  |  Steps:   92  |  Reward:    251.0  |  Flip: ✗  |  Landed: ✗  |  Rotation: 337.2°
Episode  5  |  Steps:   92  |  Reward:    251.6  |  Flip: ✗  |  Landed: ✗  |  Rotation: 338.3°
